# 02 · Limpieza y transformación (fase *Process*)

> **PRUEBA PRELIMINAR (bitácora D-09, D-11 y D-13).** Se usa la ventana **2024-11 a 2026-10, 24 meses (D-06)**. Las cifras son resultados de la prueba preliminar y deben revisarse antes de copiarse a documentos finales.

**¿Qué hace este notebook?** Toma la capa *bronze* y construye la capa *silver*: los mismos viajes, pero con fechas y números bien interpretados, y **sin los registros imposibles o sospechosos**. Cuenta cuántos viajes se excluye y por qué.

**¿Por qué?** Un viaje que dura 5 segundos o 3 días probablemente es un error o un uso atípico (bici mal devuelta, falla del sistema). Si se dejan, distorsionan los conteos. Cada regla se anota en `docs/bitacora_decisiones.md`.

In [1]:
import sys
from pathlib import Path

# Localiza la raíz del repositorio (la carpeta que contiene config/pipeline.toml) para poder importar el pipeline.
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "pipeline.toml").exists())
sys.path.insert(0, str(raiz))

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

from pipeline import bronze, config, descarga, gold, silver

cfg = config.cargar()
pd.options.display.float_format = "{:,.1f}".format
pd.options.display.max_columns = 30
pd.options.display.width = 200
print("Ventana de datos:", cfg["ventana"]["desde"], "a", cfg["ventana"]["hasta"])

Ventana de datos: 2024-11 a 2026-10


## 1. Las reglas de limpieza

Un viaje se excluye si cumple **la primera** de estas condiciones:

| Regla | Qué significa |
|---|---|
| `fecha_invalida` | La fecha u hora no se puede interpretar |
| `duracion_negativa` | El arribo es anterior al retiro |
| `menos_de_1_min` | Dura menos de la duración mínima (en `config/pipeline.toml`) |
| `mas_de_24_h` | Dura más de la duración máxima |
| `retiro_muy_antiguo` | El retiro es de antes del mes previo al del archivo |

**No se excluyen** los viajes con estaciones sin catálogo (`Temporal`, `1000`): sus movimientos son reales, solo no tienen capacidad conocida. Tampoco se toca la edad ni el género, porque no intervienen en el análisis de desbalance.

In [2]:
print("Duración mínima (s):", cfg["limpieza"]["duracion_minima_segundos"])
print("Duración máxima (h):", cfg["limpieza"]["duracion_maxima_horas"])

Duración mínima (s): 60
Duración máxima (h): 24


## 2. Aplicar las reglas y contar

El programa guarda los viajes conservados en `datos/silver/viajes.parquet` y comprueba dos cosas: que lo guardado coincide con el conteo de conservados, y que **conservados + excluidos = total de bronze**, mes por mes.

In [3]:
por_motivo, por_mes = silver.construir_silver(cfg)
por_motivo.style.format({"viajes": "{:,}", "porcentaje": "{:.3f}%"})

,motivo,viajes,porcentaje
0,conservado,"39,229,201",99.984%
1,menos_de_1_min,"5,027",0.013%
2,mas_de_24_h,"1,158",0.003%


In [4]:
por_mes.assign(pct_excluido=lambda d: 100 * d["excluidos"] / d["viajes_bronze"]).style.format(
    {"viajes_bronze": "{:,}", "viajes_silver": "{:,}", "excluidos": "{:,}", "pct_excluido": "{:.3f}%"})

,mes,viajes_bronze,viajes_silver,excluidos,pct_excluido
0,2024-11,"1,942,474","1,942,181",293,0.015%
1,2024-12,"1,680,650","1,680,413",237,0.014%
2,2025-01,"1,809,775","1,809,478",297,0.016%
3,2025-02,"1,745,620","1,745,315",305,0.017%
4,2025-03,"1,832,587","1,832,367",220,0.012%
5,2025-04,"1,727,987","1,727,701",286,0.017%
6,2025-05,"1,778,594","1,778,286",308,0.017%
7,2025-06,"1,578,804","1,578,515",289,0.018%
8,2025-07,"1,663,896","1,663,599",297,0.018%
9,2025-08,"1,610,967","1,610,684",283,0.018%


## 3. Revisión de los datos que quedan

Dos comprobaciones sobre la capa *silver*: el formato de la edad (ya como número entero) y qué porcentaje de viajes tiene sus estaciones en el catálogo.

In [5]:
con = duckdb.connect()
s = (config.SILVER / "viajes.parquet").as_posix()
print(con.execute(f"""
    SELECT count(*) AS viajes, min(edad) AS edad_min, max(edad) AS edad_max,
           sum((edad IS NULL)::INT) AS edad_vacia,
           sum((edad > 90)::INT) AS edad_mayor_90,
           sum((NOT (retiro_en_catalogo AND arribo_en_catalogo))::INT) AS viajes_con_estacion_sin_catalogo,
           min(ts_retiro) AS primer_retiro, max(ts_arribo) AS ultimo_arribo
    FROM read_parquet('{s}')""").df().T.to_string(header=False))
print()
print("Códigos de estación sin catálogo (más frecuentes):")
con.execute(f"""
    SELECT est, count(*) AS viajes FROM (
        SELECT est_retiro AS est FROM read_parquet('{s}') WHERE NOT retiro_en_catalogo
        UNION ALL SELECT est_arribo FROM read_parquet('{s}') WHERE NOT arribo_en_catalogo)
    GROUP BY 1 ORDER BY 2 DESC LIMIT 8""").df()

viajes                                       39229201
edad_min                                           16
edad_max                                          162
edad_vacia                                    1,917.0
edad_mayor_90                                 1,210.0
viajes_con_estacion_sin_catalogo              2,521.0
primer_retiro                     2024-10-31 18:19:50
ultimo_arribo                     2026-10-31 23:59:54

Códigos de estación sin catálogo (más frecuentes):


,est,viajes
0,"""Temporal 1""",968
1,"""Temporal - 2da Sección Bosque Chapultepec""",772
2,1000,548
3,Temporal 2,272
4,Temporal 1,208
5,Temporal 3,32
6,Temporal 4,31
7,"""Temporal 2""",17


**Comentario sobre la edad:** el valor máximo (162 años, y varios otros por encima de 100) es casi seguro un **error de captura** en el origen de los datos (por ejemplo, una edad mal escrita o un valor por defecto), no una persona real. Son muy pocos viajes frente a los más de 39 millones. Como la edad no interviene en el análisis de desbalance, **no se corrige ni se excluye**; solo se deja constancia (bitácora D-12).

## 4. Qué decidimos y qué falta

- Se excluyen los viajes de las reglas de arriba. El porcentaje es muy pequeño, así que **no cambia la forma del análisis**.
- Las reglas están registradas en la bitácora (D-10 y D-13) y los umbrales se pueden cambiar en `config/pipeline.toml`.
- Las edades con valores imposibles (mayores de 90) **no se corrigen**, porque la edad no se usa en este análisis.
- Los **festivos oficiales** no se excluyen aquí: se descartan como días laborables en la capa *gold* (D-13).

**Siguiente paso:** `03_analisis.ipynb` calcula el desbalance por estación.